In [1]:
!pip install Sastrawi

In [2]:
import re
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


In [4]:
# 5 dokumen berbahasa Indonesia bertema pengumuman kampus / teknologi
# (dibuat sendiri, meniru gaya pengumuman resmi Fakultas Teknik)

documents = [
    "Fakultas Teknik UNM mengumumkan pelaksanaan seminar nasional "
    "teknologi digital yang akan diselenggarakan pada tanggal 15 Oktober "
    "2026 di aula utama fakultas dan terbuka bagi seluruh mahasiswa.",

    "Mahasiswa yang mengikuti program pertukaran pelajar diwajibkan "
    "mengunggah dokumen persyaratan melalui portal akademik sebelum "
    "batas akhir pendaftaran pada tanggal 25 September 2026.",

    "Perpustakaan Fakultas Teknik mulai menyediakan layanan peminjaman "
    "buku berbasis kode QR untuk mempermudah mahasiswa dalam melakukan "
    "pencatatan dan pengembalian koleksi secara mandiri.",

    "Kelompok mahasiswa Teknik Komputer menyelenggarakan pameran "
    "proyek teknologi yang menampilkan berbagai perangkat berbasis "
    "mikrokontroler, robotika, dan Internet of Things.",

    "Kampus mengimbau seluruh mahasiswa untuk berhati-hati terhadap "
    "pesan elektronik yang meminta data pribadi atau kode verifikasi "
    "karena dapat menjadi upaya penipuan digital.",
]

print(f"Jumlah dokumen: {len(documents)}\n")


Jumlah dokumen: 5



In [5]:
stopword_factory = StopWordRemoverFactory()
stopword_list = set(stopword_factory.get_stop_words())

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()


In [6]:
def preprocess_text(text):
    """
    Pipeline preprocessing teks Bahasa Indonesia:
    1. Case folding
    2. Cleaning (hapus angka, tanda baca, karakter khusus)
    3. Tokenisasi
    4. Stopwords removal (Sastrawi)
    5. Stemming (Sastrawi)

    Return:
        tokens_before (list) : token hasil tokenisasi awal (sebelum stopword & stemming)
        tokens_after  (list) : token akhir setelah full pipeline
    """
    # 1. Case folding
    text = text.lower()

    # 2. Cleaning: hapus angka, tanda baca, dan karakter khusus (sisakan huruf & spasi)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 3. Tokenisasi (split spasi, karena teks sudah bersih dari tanda baca)
    tokens_before = text.split()

    # 4. Stopwords removal
    tokens_no_stopwords = [t for t in tokens_before if t not in stopword_list]

    # 5. Stemming (Sastrawi stemmer bekerja per-kata)
    tokens_after = [stemmer.stem(t) for t in tokens_no_stopwords]
    # Buang string kosong hasil stemming jika ada
    tokens_after = [t for t in tokens_after if t != ""]

    return tokens_before, tokens_after


In [7]:
results = []
for i, doc in enumerate(documents, 1):
    tokens_before, tokens_after = preprocess_text(doc)
    results.append({
        "No": i,
        "Dokumen Asli": doc,
        "Tokens Sebelum": tokens_before,
        "Tokens Sesudah": tokens_after,
        "Jumlah Token Sebelum": len(tokens_before),
        "Jumlah Token Sesudah": len(tokens_after),
    })


In [8]:
print("=" * 80)
print("PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING (2 Dokumen Contoh)")
print("=" * 80)

for r in results[:2]:
    print(f"\n--- Dokumen {r['No']} ---")
    print("Teks asli   :", r["Dokumen Asli"])
    print("Sebelum (token hasil tokenisasi) :")
    print(" ", r["Tokens Sebelum"])
    print("Sesudah (stopword removal + stemming) :")
    print(" ", r["Tokens Sesudah"])


PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING (2 Dokumen Contoh)

--- Dokumen 1 ---
Teks asli   : Fakultas Teknik UNM mengumumkan pelaksanaan seminar nasional teknologi digital yang akan diselenggarakan pada tanggal 15 Oktober 2026 di aula utama fakultas dan terbuka bagi seluruh mahasiswa.
Sebelum (token hasil tokenisasi) :
  ['fakultas', 'teknik', 'unm', 'mengumumkan', 'pelaksanaan', 'seminar', 'nasional', 'teknologi', 'digital', 'yang', 'akan', 'diselenggarakan', 'pada', 'tanggal', 'oktober', 'di', 'aula', 'utama', 'fakultas', 'dan', 'terbuka', 'bagi', 'seluruh', 'mahasiswa']
Sesudah (stopword removal + stemming) :
  ['fakultas', 'teknik', 'unm', 'umum', 'laksana', 'seminar', 'nasional', 'teknologi', 'digital', 'selenggara', 'tanggal', 'oktober', 'aula', 'utama', 'fakultas', 'buka', 'seluruh', 'mahasiswa']

--- Dokumen 2 ---
Teks asli   : Mahasiswa yang mengikuti program pertukaran pelajar diwajibkan mengunggah dokumen persyaratan melalui portal akademik sebelum batas akhir pendafta

In [9]:
print("\n" + "=" * 80)
print("STATISTIK JUMLAH TOKEN")
print("=" * 80)

stat_rows = []
for r in results:
    before = r["Jumlah Token Sebelum"]
    after = r["Jumlah Token Sesudah"]
    reduction_pct = (before - after) / before * 100 if before > 0 else 0
    stat_rows.append({
        "Dokumen": f"Dok {r['No']}",
        "Token Sebelum": before,
        "Token Sesudah": after,
        "Pengurangan (%)": round(reduction_pct, 2),
    })

df_stats = pd.DataFrame(stat_rows)
total_before = df_stats["Token Sebelum"].sum()
total_after = df_stats["Token Sesudah"].sum()
total_reduction = (total_before - total_after) / total_before * 100

df_stats.loc[len(df_stats)] = [
    "TOTAL", total_before, total_after, round(total_reduction, 2)
]

print(df_stats.to_string(index=False))



STATISTIK JUMLAH TOKEN
Dokumen  Token Sebelum  Token Sesudah  Pengurangan (%)
  Dok 1             24             18            25.00
  Dok 2             20             17            15.00
  Dok 3             22             18            18.18
  Dok 4             19             17            10.53
  Dok 5             23             17            26.09
  TOTAL            108             87            19.44


In [10]:
print("\n" + "=" * 80)
print("HASIL PREPROCESSING SELURUH DOKUMEN (RINGKAS)")
print("=" * 80)
for r in results:
    print(f"\nDok {r['No']} - sebelum: {r['Jumlah Token Sebelum']} token, "
          f"sesudah: {r['Jumlah Token Sesudah']} token")
    print("  ->", r["Tokens Sesudah"])



HASIL PREPROCESSING SELURUH DOKUMEN (RINGKAS)

Dok 1 - sebelum: 24 token, sesudah: 18 token
  -> ['fakultas', 'teknik', 'unm', 'umum', 'laksana', 'seminar', 'nasional', 'teknologi', 'digital', 'selenggara', 'tanggal', 'oktober', 'aula', 'utama', 'fakultas', 'buka', 'seluruh', 'mahasiswa']

Dok 2 - sebelum: 20 token, sesudah: 17 token
  -> ['mahasiswa', 'ikut', 'program', 'tukar', 'ajar', 'wajib', 'unggah', 'dokumen', 'syarat', 'lalu', 'portal', 'akademik', 'batas', 'akhir', 'daftar', 'tanggal', 'september']

Dok 3 - sebelum: 22 token, sesudah: 18 token
  -> ['pustaka', 'fakultas', 'teknik', 'mulai', 'sedia', 'layan', 'pinjam', 'buku', 'bas', 'kode', 'qr', 'mudah', 'mahasiswa', 'laku', 'catat', 'kembali', 'koleksi', 'mandiri']

Dok 4 - sebelum: 19 token, sesudah: 17 token
  -> ['kelompok', 'mahasiswa', 'teknik', 'komputer', 'selenggara', 'pamer', 'proyek', 'teknologi', 'tampil', 'bagai', 'perangkat', 'bas', 'mikrokontroler', 'robotika', 'internet', 'of', 'things']

Dok 5 - sebelum: 23 

Hasil analisis:
Tahapan preprocessing yang dilakukan pada kelima dokumen, yaitu case folding, cleaning, tokenisasi, stopword removal, dan stemming, berhasil mengurangi jumlah token dari 108 menjadi 87 token atau mengalamai reduksi rata-rata sebesar 19,44%. Pengurangan ini membantu meningkatkan efisiensi sistem Informasi Retrieval (IR) karena jumlah data yang diproses menjadi lebih sedikit, variasi bentuk kata dapat diseragamkan, serta kata-kata yang kurang memiliki makna penting dapat dihilangkan. Namun, stremming yang terllau agresif, seperti kata "berbasis" yang berubah menjadi "bas", dapat menyebabkan perubahan makna dan berpengaruh terhadap ketepatan hasil pencarian.